In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilează circuite de la distanță cu Qiskit Transpiler Service

> **Danger:** Începând cu 18 iulie 2025, serviciul este în curs de migrare pentru a suporta noua platformă IBM Quantum&reg; și nu este disponibil. Pentru pasele AI, poți folosi [modul local](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Serviciul este o versiune beta, supusă modificărilor.
>     Dacă ai feedback sau dorești să contactezi echipa de dezvoltare, folosește acest canal din [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Qiskit Transpiler Service oferă capabilități de transpilare în cloud. Pe lângă capabilitățile locale ale Transpiler-ului Qiskit, sarcinile tale de transpilare pot beneficia atât de resursele cloud IBM Quantum, cât și de pasele de transpilare bazate pe AI.

Qiskit Transpiler Service oferă o bibliotecă Python pentru a integra fără dificultăți acest serviciu și capabilitățile sale în fluxurile și tiparele tale Qiskit existente. Acest serviciu este disponibil doar pentru utilizatorii planurilor IBM Quantum Premium, Flex și On-Prem (prin IBM Quantum Platform API).

<span id="install-transpiler-service"></span>
## Instalează pachetul qiskit-ibm-transpiler
Pentru a folosi Qiskit Transpiler Service, instalează pachetul `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Pachetul se autentifică automat folosind [credențialele tale de pe IBM Quantum Platform](/guides/cloud-setup), în concordanță cu modul în care [Qiskit Runtime le gestionează](/guides/initialize-account):
- Variabilă de mediu: `QISKIT_IBM_TOKEN`
- Fișierul de configurare `~/.qiskit/qiskit-ibm.json` (sub secțiunea `default-ibm-quantum`).

*Notă*: Acest pachet necesită Qiskit SDK v1.X.

## Opțiunile de transpilare ale qiskit-ibm-transpiler
- `backend_name` (opțional, str) - Un nume de Backend, așa cum ar fi așteptat de QiskitRuntimeService (de exemplu, `ibm_torino`). Dacă este setat, metoda de transpilare folosește layout-ul din Backend-ul specificat pentru operațiunea de transpilare. Dacă orice altă opțiune care afectează aceste setări este configurată, cum ar fi `coupling_map`, setările `backend_name` sunt suprascrise.
- `coupling_map` (opțional, List[List[int]]) - O listă validă de coupling map (de exemplu, [[0,1],[1,2]]). Dacă este setată, metoda de transpilare folosește acest coupling map pentru operațiunea de transpilare. Dacă este definit, suprascrie orice valoare specificată pentru `target`.
- `optimization_level` (int) - Nivelul potențial de optimizare care se aplică în timpul procesului de transpilare. Valorile valide sunt [1,2,3], unde 1 reprezintă cea mai mică optimizare (și cea mai rapidă), iar 3 reprezintă cea mai mare optimizare (și cea mai intensivă din punct de vedere al timpului).
- `ai` ("true", "false", "auto") - Dacă să se folosească capabilități bazate pe AI în timpul transpilării. Capabilitățile bazate pe AI disponibile pot fi pentru pasele de transpilare `AIRouting` sau alte metode de sinteză bazate pe AI. Dacă această valoare este `"true"`, serviciul aplică diferite pase de transpilare bazate pe AI în funcție de `optimization_level` solicitat. Dacă este `"false"`, folosește cele mai recente funcționalități de transpilare Qiskit fără AI. În final, dacă este `"auto"`, serviciul decide dacă să aplice pasele euristice standard Qiskit sau pasele bazate pe AI în funcție de Circuit-ul tău.
- `qiskit_transpile_options` (dict) - Un obiect dicționar Python care poate include orice altă opțiune validă în [metoda Qiskit `transpile()`](defaults-and-configuration-options). Dacă `qiskit_transpile_options` include `optimization_level`, acesta este ignorat în favoarea `optimization_level` specificat ca parametru de intrare. Dacă `qiskit_transpile_options` include orice opțiune nerecunoscută de metoda Qiskit `transpile()`, biblioteca generează o eroare.

Pentru mai multe informații despre metodele disponibile ale `qiskit-ibm-transpiler`, consultă [referința API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Pentru a afla mai multe despre API-ul serviciului, consultă [documentația REST API a Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Exemple
Exemplele următoare demonstrează cum să transpilezi circuite folosind Qiskit Transpiler Service cu diferiți parametri.

1. Creează un Circuit și apelează Qiskit Transpiler Service pentru a transpila Circuit-ul cu `ibm_torino` ca `backend_name`, 3 ca `optimization_level`, și fără a folosi AI în timpul transpilării.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Notă*: poți folosi doar dispozitive `backend_name` la care ai acces cu contul tău IBM Quantum. Pe lângă `backend_name`, `TranspilerService` permite și `coupling_map` ca parametru.

2. Creează un Circuit similar și transpilează-l, solicitând capabilități de transpilare AI prin setarea flag-ului `ai` la `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Creează un Circuit similar și transpilează-l, lăsând serviciul să decidă dacă să folosească pasele de transpilare bazate pe AI.